In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
!pip install -q youtube-transcript-api langchain-community langchain-google-genai faiss-cpu tiktoken python-dotenv


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: C:\Users\MSI\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [3]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

C:\Users\MSI\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\MSI\AppData\Local\Temp\ipykernel_17512\3933634437.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
video_id = "bR6rdchGjDY"

try:
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=["hi"])

    transcript = " ".join(chunk.text for chunk in transcript_list)
    # print(type(transcript_list))
    print(transcript)

except TranscriptsDisabled:
    print("No caption avalable for this video")


आई कैन नो यू वाकिंग इफ यू आ साइड दिस माय हार्ट रे अराउंड लाइक ओशन इन माय बिकज़ दे आर सो मेनी थिंग्स आई लेफ्ट आई एम से आई कैन फील यू आई नो रियलिटी इफ यू आर लाइक साइड दिस हेलो गाइस तो कैसे हैं? बहुत-बहुत स्वागत है आपका मेरे स्ट्रीम में। थैंक्स फॉर जॉइन माय स्ट्रीम गाइस। हाउ आर यू ऑल? हेलो युवराज। हेलो समर। कैसे हैं आप लोग? बहुत-बहुत स्वागत है आपका मेरे स्ट्रीम में। थैंक्स फॉर जॉइ स्ट्रीम। हाय ईजी। हेलो युवराज। हेलो समर। बस आपकी लाइव का ही वेट कर रहा था। थैंक्स सर। हाउ आर यू? आई एम ऑल गुड समर। व्हाट अबाउट यू? पांच बार देखी कि लाइव ऑन है कि नहीं। अरे यार सो स्वीट। लाइक करो गाइस। समर भाई पंजाब विो किधर किधर हो तसी? पटियाला। हाय जी। हेलो हाउ आर यू? आई एम ऑल गुड। समर व्हाट अबाउट यू? आई एम फाइन तो मैं भी फाइन। अरे यार मेरे डायलॉग्स कॉपी करता है बदतमीज। कोई बात नहीं। गाइस एक्चुअली मेरी आवाज बहुत ही घटिया हो चुकी है आज जस्ट बिकॉज़ मुझे सर्दी है एंड मेरी नाक बंद हो चुकी है तो खैर कोई बात नहीं तबीयत कैसी है अब बस खराब है यार हेलो मानव कैसे हैं आपका स्वागत है आपका स्ट्रीम में थैंक्स फॉर जॉइनिंग द स्ट

In [9]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 3000, chunk_overlap=300)
chunks = splitter.create_documents([transcript])

print(len(chunks))

16


In [10]:
embeddings = GoogleGenerativeAIEmbeddings(model = "models/gemini-embedding-001")
vector_store = FAISS.from_documents(chunks, embeddings)

In [11]:
vector_store.index_to_docstore_id

{0: 'cfcebdd7-e451-436b-96c7-60d3bb0596c6',
 1: '1f35ffbb-8bc2-4aef-8557-8fb04d32b54e',
 2: '04015d25-2474-4269-b221-99b764f54eee',
 3: 'a695981c-50f6-4791-8f76-2b5085ccfc84',
 4: 'cc2a4b8c-a746-40d8-8e83-611d4d623973',
 5: '567b7e9f-f3a7-45a2-a6ad-331981a6d431',
 6: 'b7074487-add1-47c1-bf19-dca020ad1b20',
 7: '93716006-ae30-4d5e-bec3-9d7a737e7c88',
 8: 'f8ec5b0a-22d5-483c-ac51-42a7fa6a4f44',
 9: 'c22fc2d9-09d0-4c7e-a4ff-c46189132db3',
 10: 'cd9bffd9-b1fe-4cb8-8116-e34442abd843',
 11: '18afd944-66aa-4983-9007-6d0dfdea3696',
 12: 'dc32d11f-806b-4ced-8af4-1f2d2dd5bc84',
 13: 'b6d64365-afad-492a-9d62-d156f1587369',
 14: '7bef68c4-4db6-4aa6-8998-0f03aadc77b0',
 15: '501656dd-2415-45dc-997c-2c895f407e2c'}

In [12]:
vector_store.get_by_ids(['9291fe55-9c97-47b5-aa9d-e6ed733eaf8c'])

[]

In [13]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [14]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000222346FDD30>, search_kwargs={'k': 3})

In [15]:
retriever.invoke("Should we take youtube full url or only youtube specific code?")

[Document(id='a695981c-50f6-4791-8f76-2b5085ccfc84', metadata={}, page_content='श्योर यहां पे आप अपना की पेस्ट करो। उसके बाद आपको कुछ लाइब्रेरीज इंस्टॉल करनी है। इस कोड को आपको एज इट इज रन कर देना है। यहां पर हमने जो भी नेसेसरी इंपोर्ट्स हैं वो ले लिए हैं। और यहां से अब मेन काम शुरू होता है। तो जैसा मैंने बताया हम वो सेम रैग आर्किटेक्चर यूज़ करेंगे जो मैंने आपको थोड़ी देर पहले डायग्राम में दिखाया है। और सबसे पहले हम लोग इंडेक्सिंग वाला स्टेप कंप्लीट करेंगे। ठीक है? इंडेक्सिंग में हम सबसे पहले टारगेट करेंगे स्टेप वन पे दैट इज हम YouTube की एपीआई को यूज़ करके किसी भी YouTube वीडियो का ट्रांसक्रिप्ट लोड करेंगे और उसको एज स्ट्रिंग अपने कोड में लेके आएंगे। सो यहां पर यह एक कोड लिखा हुआ है जो इंटरनली YouTube ट्रांसक्रिप्ट एपीआई को यूज़ करता है। ठीक है? जैसा कि मैंने बोला यह करने के और भी कई तरीके हैं। बट यह तरीका मुझे अभी सबसे सही लगा। बिकॉज़ यह हर तरह के वीडियो के साथ मुझे सही रिजल्ट्स दे रहा था। जैसे मान लो मेरे पास यह वीडियो है 3 ब्लू वन ब्राउन का और मुझे इस वीडियो का ट्रांसक्रिप्ट लोड करना ह

In [16]:
llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0.3)

In [19]:
prompt = PromptTemplate(
    template="""
        You are a Helpful Assistant
        Answer only from provided transcript Context in English Only
        if the context is inefficient , just say you dont know.

        {context},
        Question:{question}
    """,
    input_variables=["context", "question"]
)

In [21]:
question = "Should we take youtube full url or only youtube specific code?"
retrieved_docs = retriever.invoke(question)

In [22]:
context_text="\n\n".join(doc.page_content for doc in retrieved_docs)

In [23]:
final_prompt = prompt.invoke({"context": context_text,"question": question})

final_prompt

StringPromptValue(text='\n        You are a Helpful Assistant\n        Answer only from provided transcript Context in English Only\n        if the context is inefficient , just say you dont know.\n\n        श्योर यहां पे आप अपना की पेस्ट करो। उसके बाद आपको कुछ लाइब्रेरीज इंस्टॉल करनी है। इस कोड को आपको एज इट इज रन कर देना है। यहां पर हमने जो भी नेसेसरी इंपोर्ट्स हैं वो ले लिए हैं। और यहां से अब मेन काम शुरू होता है। तो जैसा मैंने बताया हम वो सेम रैग आर्किटेक्चर यूज़ करेंगे जो मैंने आपको थोड़ी देर पहले डायग्राम में दिखाया है। और सबसे पहले हम लोग इंडेक्सिंग वाला स्टेप कंप्लीट करेंगे। ठीक है? इंडेक्सिंग में हम सबसे पहले टारगेट करेंगे स्टेप वन पे दैट इज हम YouTube की एपीआई को यूज़ करके किसी भी YouTube वीडियो का ट्रांसक्रिप्ट लोड करेंगे और उसको एज स्ट्रिंग अपने कोड में लेके आएंगे। सो यहां पर यह एक कोड लिखा हुआ है जो इंटरनली YouTube ट्रांसक्रिप्ट एपीआई को यूज़ करता है। ठीक है? जैसा कि मैंने बोला यह करने के और भी कई तरीके हैं। बट यह तरीका मुझे अभी सबसे सही लगा। बिकॉज़ यह हर तरह के वीडियो के साथ 

In [24]:
answer = llm.invoke(final_prompt)
print(answer.content)

You should take only the YouTube video ID, not the full URL.


In [107]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [108]:
def format_doc(retrieved_doc):
    context_text="\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [113]:
parallel_chain = RunnableParallel({
    'context':retriever|RunnableLambda(format_doc),
    'question': RunnablePassthrough()
})

In [114]:
parallel_chain.invoke('How Earth looks from space?')

{'context': "bumped up into each other and and made the you know made the mountains. Um and then the coast is obviously cool because you can see particularly on the east side uh when the rivers run off how they interact with the water on that side, the different colors and swirls from that side. And then at one point in time I took a picture of the interaction between India and how do we call it nowadays? Sri Lanka. >> Yeah. >> Um um and it seems like there's a connection there. I'll just say that. [laughter] >> I'll say that. I think I think that picture took a there was a little bit of a discussion about that picture, but it it looks like there is you know they absolutely could be connected. >> Yeah, there must be some >> there's something there. >> There's something connected. [laughter] Maybe we leave it out there like that. >> Yeah. Yeah. Leave it out there. >> What's And when you look out of the window, okay, this is my sci-fi brain talking, okay? So, just don't doubt me on this 

In [115]:
parser = StrOutputParser()

In [116]:
main_chain = parallel_chain | prompt | llm | parser

In [118]:
main_chain.invoke('Can you summarize the video?')

'The video transcript features an astronaut discussing various aspects of her experiences in space. She describes observations of Earth from space, including the amazing colors of India, the Himalayas, the interaction of rivers with coastal waters, and new sightings of fishing boats with bright lights at night off the west coast of India. She notes how India\'s cities are spectacularly lit up at night, resembling connected nerves.\n\nThe astronaut also shares personal experiences and challenges:\n*   **Unexpected Mission Length:** A 10-day test flight turned into a 9-month mission due to a booster failure, leading to a prolonged stay in space.\n*   **Risks and Dangers:** She recounts being woken up due to a satellite exploding below them, creating a debris field, and the crew having to go to their "safe haven" spacecraft. She also mentions the hundreds of situations they train for, including potential failures.\n*   **Emotional Aspects:** She discusses how fellow astronauts become like